# vulnrag — интерактивные запросы

Пиши вопрос про уязвимость компонента и получай вердикт + список CVE + ответ модели.

**Перед запуском:** должны быть подняты Qdrant и Ollama (`bge-m3`, `qwen2.5:3b`), а база наполнена (`./run.sh sync`). Ядро — `vulnrag (venv)` или `venv/bin/python`.

Порядок: выполни ячейку **setup** (один раз), потом пользуйся либо полем ввода, либо функцией `ask("...")`.

In [1]:
# setup — выполнить один раз
from vulnrag.factory import build_components
from vulnrag.query.answer import answer

store, emb, llm = build_components()   # реальный Qdrant + Ollama
print('Готово. В базе CVE:', store.count())

_LABEL = {
    'vulnerable':   '\U0001F534 УЯЗВИМО',
    'unconfirmed':  '\U0001F7E1 НЕ ПОДТВЕРЖДЕНО (укажи версию)',
    'safe':         '\U0001F7E2 БЕЗОПАСНО',
    'not_found':    '\U000026AA НЕ НАЙДЕНО В БАЗЕ',
}

def ask(question: str):
    """Задать вопрос и красиво напечатать результат."""
    r = answer(question, embedder=emb, store=store, llm=llm)
    print('Вопрос :', question)
    print('Вердикт:', _LABEL.get(r.verdict, r.verdict), f'(lang={r.lang})')
    if r.cves:
        print('CVE:')
        for c in r.cves:
            print(f"  \u2022 {c['cve_id']}  severity={c.get('severity')}  fixed={c.get('fixed_versions')}")
            for ref in (c.get('references') or [])[:2]:
                print('      ', ref)
    print('\n--- Ответ модели ---')
    print(r.answer_text)
    return r

Готово. В базе CVE: 19952


In [12]:
ask('Безопасен ли samtools 1.18?')

Вопрос : Безопасен ли samtools 1.18?
Вердикт: 🔴 УЯЗВИМО (lang=ru)
CVE:
  • CVE-2026-31973  severity=HIGH  fixed=[]
       https://github.com/samtools/samtools/commit/06fc2a219b3d7c94d3f412c09f6d1efd51199f2f
       https://github.com/samtools/samtools/security/advisories/GHSA-x86f-q6fj-cm43
  • CVE-2026-31972  severity=CRITICAL  fixed=[]
       https://github.com/samtools/samtools/commit/3036eb9af945fcef359427a2d359855553da4adf
       https://github.com/samtools/samtools/security/advisories/GHSA-72c8-4jf3-f27p

--- Ответ модели ---
vulnerable (CVE-2026-31972, CVE-2026-31973), 
remediation: Upgrade to fixed version 1.21.1 or later. Mitigations can be found in the references provided.


QueryResult(verdict='vulnerable', answer_text='vulnerable (CVE-2026-31972, CVE-2026-31973), \nremediation: Upgrade to fixed version 1.21.1 or later. Mitigations can be found in the references provided.', cves=[{'cve_id': 'CVE-2026-31973', 'severity': 'HIGH', 'fixed_versions': [], 'references': ['https://github.com/samtools/samtools/commit/06fc2a219b3d7c94d3f412c09f6d1efd51199f2f', 'https://github.com/samtools/samtools/security/advisories/GHSA-x86f-q6fj-cm43', 'http://www.openwall.com/lists/oss-security/2026/03/18/12']}, {'cve_id': 'CVE-2026-31972', 'severity': 'CRITICAL', 'fixed_versions': [], 'references': ['https://github.com/samtools/samtools/commit/3036eb9af945fcef359427a2d359855553da4adf', 'https://github.com/samtools/samtools/security/advisories/GHSA-72c8-4jf3-f27p', 'http://www.openwall.com/lists/oss-security/2026/03/18/11']}], lang='ru')

## Поле ввода с кнопкой
Введи вопрос и нажми **Спросить** (LLM на CPU отвечает ~3–30 с).

In [6]:
import ipywidgets as W
from IPython.display import display

_box = W.Text(value='Безопасен ли openssl 3.0.1?',
              placeholder='напр. Безопасен ли log4j 2.14.1?',
              description='Запрос:', layout=W.Layout(width='80%'))
_btn = W.Button(description='Спросить', button_style='primary')
_out = W.Output()

def _run(_=None):
    _out.clear_output()
    with _out:
        ask(_box.value)

_btn.on_click(_run)
display(W.VBox([W.HBox([_box, _btn]), _out]))